In [ ]:
!pip install -U transformers -q
!pip install -q supervision -q
!pip uninstall -y onnxruntime onnxruntime-gpu -q
!pip install -q 'onnxruntime-gpu==1.19.2' -q
print('Done — restart runtime if this is the first run')

## Local Inference on GPU
Model page: https://huggingface.co/MBARI-org/mbari-uav-vit-b-16

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/MBARI-org/mbari-uav-vit-b-16)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("image-classification", model="MBARI-org/mbari-uav-vit-b-16")
pipe("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/hub/parrots.png")

In [ ]:
# Load model directly
from transformers import AutoImageProcessor, AutoModelForImageClassification

processor = AutoImageProcessor.from_pretrained("MBARI-org/mbari-uav-vit-b-16")
model = AutoModelForImageClassification.from_pretrained("MBARI-org/mbari-uav-vit-b-16")

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

In [ ]:
# Allow access to personal google drive and add new folders

# Connect Google Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True) # This will prompt for authorization.

# This will create the uavs files if they don't exist.
folders =  ["uavs-rfdetr/"]
for folder in folders:
  path = "/content/drive/MyDrive/" + folder
  if not os.path.exists(path): # Create the folder if it does not exist
    os.mkdir(path)

In [ ]:
!mkdir {HOME}/datasets
%cd {HOME}/datasets
from google.colab import userdata

!cp "/content/drive/MyDrive/uavs-rfdetr/onnx_export/rfdetr-nano.onnx" "/content/rfdetr-nano.onnx"

!mkdir -p /content/testing/
!cp -r "/content/drive/MyDrive/uavs-rfdetr/testing/images.tar.gz" "/content/testing/"

!cp -r "/content/drive/MyDrive/uavs-rfdetr/testing/labels.tar.gz" "/content/testing/"

!tar xf /content/testing/images.tar.gz --directory /content/testing/

!tar xf /content/testing/labels.tar.gz --directory /content/testing/

!ls /content/testing/

In [ ]:
## Find first test image and load its ground-truth YOLO labels

import os, yaml
import numpy as np
import supervision as sv
from PIL import Image

IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')

# Optionally load detector class names from data.yaml on Drive for GT labels
_yaml_path = '/content/drive/MyDrive/uavs-rfdetr/testing/data.yaml'
if os.path.exists(_yaml_path):
    with open(_yaml_path) as f:
        _y = yaml.safe_load(f)
    _names = _y.get('names', [])
    DETECTOR_CLASSES = list(_names.values()) if isinstance(_names, dict) else list(_names)
    print(f'Detector classes ({len(DETECTOR_CLASSES)}): {DETECTOR_CLASSES}')
else:
    DETECTOR_CLASSES = None
    print('data.yaml not found — GT boxes will show class IDs')

# Walk /content/testing to find images and labels
def _walk_files(root, exts):
    paths = []
    for dirpath, _, files in os.walk(root):
        for f in sorted(files):
            if f.lower().endswith(exts):
                paths.append(os.path.join(dirpath, f))
    return sorted(paths)

test_image_paths = _walk_files('/content/testing', IMAGE_EXTS)
print(f'\nFound {len(test_image_paths)} test images')
first_image_path = test_image_paths[0]
print(f'Using: {first_image_path}')

# Find labels directory (first dir containing .txt files)
labels_dir = None
for dirpath, _, files in os.walk('/content/testing'):
    if any(f.endswith('.txt') for f in files):
        labels_dir = dirpath
        break
print(f'Labels dir: {labels_dir}')

# Load YOLO ground truth for this image.
# Label files are named <image_filename>.txt  (e.g. foo.JPG.txt), not <stem>.txt.
# class_id is forced to 0 (class-agnostic) so GT matches the single-class detector output.
def load_yolo_gt(img_path, lbl_dir, img_w, img_h):
    if lbl_dir is None:
        return sv.Detections.empty()
    fname    = os.path.basename(img_path)
    lbl_path = os.path.join(lbl_dir, fname + '.txt')
    if not os.path.exists(lbl_path):
        print(f'No label file found: {lbl_path}')
        return sv.Detections.empty()
    xyxy, class_ids = [], []
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cx, cy, bw, bh = map(float, parts[1:5])
            x1 = max(0.0, (cx - bw / 2) * img_w)
            y1 = max(0.0, (cy - bh / 2) * img_h)
            x2 = min(float(img_w), (cx + bw / 2) * img_w)
            y2 = min(float(img_h), (cy + bh / 2) * img_h)
            xyxy.append([x1, y1, x2, y2])
            class_ids.append(0)  # class-agnostic: detector outputs class 0 for every box
    if not xyxy:
        return sv.Detections.empty()
    return sv.Detections(
        xyxy=np.array(xyxy, dtype=np.float32),
        class_id=np.array(class_ids, dtype=int),
    )

image_pil = Image.open(first_image_path).convert('RGB')
img_w, img_h = image_pil.size
gt = load_yolo_gt(first_image_path, labels_dir, img_w, img_h)
print(f'Ground truth boxes: {len(gt)}')

In [ ]:
## Load ONNX detector + ViT classifier

import torch, numpy as np, onnxruntime as ort, supervision as sv
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForImageClassification

ONNX_MODEL_PATH      = '/content/rfdetr-nano.onnx'
IMAGE_SIZE           = 384
DEVICE               = 'cuda' if torch.cuda.is_available() else 'cpu'
SWEEP_BASE_THRESHOLD = 0.10
SWEEP_THRESHOLDS     = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]


class RFDetrOnnxModel:
    """RF-DETR ONNX inference — full image, no tiling.
    ONNX outputs:
      dets:   [1, 300, 4]  normalised cx,cy,w,h
      labels: [1, 300, 1]  raw logits (sigmoid -> confidence)
    """

    def __init__(self, model_path, image_size=384):
        providers = (
            ['CUDAExecutionProvider', 'CPUExecutionProvider']
            if torch.cuda.is_available() else ['CPUExecutionProvider']
        )
        self.session      = ort.InferenceSession(model_path, providers=providers)
        self.input_name   = self.session.get_inputs()[0].name
        self.output_names = [o.name for o in self.session.get_outputs()]
        self.image_size   = image_size
        print(f'Active provider : {self.session.get_providers()[0]}')
        print(f'ONNX inputs     : {self.input_name}')
        print(f'ONNX outputs    : {self.output_names}')

    def infer(self, image_pil: Image.Image, conf_threshold: float) -> sv.Detections:
        img_w, img_h = image_pil.size
        tile = image_pil.resize((self.image_size, self.image_size), Image.BILINEAR)
        tile = np.array(tile, dtype=np.float32) / 255.0
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        tile = ((tile - mean) / std).transpose(2, 0, 1)[None]
        dets, labels = self.session.run(self.output_names, {self.input_name: tile})
        dets   = dets[0]    # [300, 4]
        labels = labels[0]  # [300, 1]
        scores = 1.0 / (1.0 + np.exp(-labels[:, 0]))
        xyxy, confs = [], []
        for box, score in zip(dets, scores):
            if score < conf_threshold:
                continue
            cx, cy, bw, bh = box
            x1 = max(0.0, (cx - bw / 2) * img_w)
            y1 = max(0.0, (cy - bh / 2) * img_h)
            x2 = min(float(img_w), (cx + bw / 2) * img_w)
            y2 = min(float(img_h), (cy + bh / 2) * img_h)
            if x2 <= x1 or y2 <= y1:
                continue
            xyxy.append([x1, y1, x2, y2])
            confs.append(float(score))
        if not xyxy:
            return sv.Detections.empty()
        return sv.Detections(
            xyxy=np.array(xyxy, dtype=np.float32),
            confidence=np.array(confs, dtype=np.float32),
            class_id=np.zeros(len(xyxy), dtype=int),
        )


detection_model = RFDetrOnnxModel(ONNX_MODEL_PATH, image_size=IMAGE_SIZE)
print('Detection model ready\n')

# ── ViT classifier ─────────────────────────────────────────────────────────
clf_processor = AutoImageProcessor.from_pretrained('MBARI-org/mbari-uav-vit-b-16')
clf_model     = AutoModelForImageClassification.from_pretrained(
                    'MBARI-org/mbari-uav-vit-b-16').to(DEVICE)
clf_model.eval()
id2label = clf_model.config.id2label
print(f'Classifier — device: {DEVICE}  classes ({len(id2label)}): {list(id2label.values())}')

# ── ONNX coordinate space sanity check ────────────────────────────────────
_tile = np.array(Image.open(test_image_paths[0]).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR), dtype=np.float32) / 255.0
_mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
_std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
_tile = ((_tile - _mean) / _std).transpose(2, 0, 1)[None]
_raw_dets, _raw_labels = detection_model.session.run(detection_model.output_names, {detection_model.input_name: _tile})
_raw_scores = 1.0 / (1.0 + np.exp(-_raw_labels[0, :, 0]))
_top = _raw_dets[0][_raw_scores.argmax()]
print(f'\nONNX coord check — top box cx/cy/w/h: {_top}')
if _top.max() > 2.0:
    print('  WARNING: values > 2 suggest absolute pixel coords')
else:
    print('  OK: values look normalised [0,1]')

In [ ]:
## Detection loop — full image inference on all test images (no SAHI)

from tqdm import tqdm

all_raw_preds = []   # sv.Detections per image at base threshold
all_gt        = []   # sv.Detections per image (ground truth)

for img_path in tqdm(test_image_paths, desc='Detecting'):
    img = Image.open(img_path).convert('RGB')
    w, h = img.size
    all_raw_preds.append(detection_model.infer(img, conf_threshold=SWEEP_BASE_THRESHOLD))
    all_gt.append(load_yolo_gt(img_path, labels_dir, w, h))

n_imgs = len(all_raw_preds)
n_dets = sum(len(d) for d in all_raw_preds)
n_gt   = sum(len(g) for g in all_gt)
print(f'\n{n_imgs} images  |  {n_dets} candidate detections  |  {n_gt} GT boxes')

In [ ]:
## Stage 2 — classify every candidate crop once across all images

from tqdm import tqdm

all_clf_labels = []   # list[list[str]]  — one inner list per image
all_clf_scores = []   # list[list[float]]

for img_path, det in tqdm(zip(test_image_paths, all_raw_preds),
                          total=len(test_image_paths), desc='Classifying'):
    img = Image.open(img_path).convert('RGB')
    img_labels, img_scores = [], []
    for x1, y1, x2, y2 in det.xyxy.astype(int):
        if x2 <= x1 or y2 <= y1:   # degenerate box after int truncation — skip
            img_labels.append('invalid')
            img_scores.append(0.0)
            continue
        crop   = img.crop((x1, y1, x2, y2))
        inputs = clf_processor(images=crop, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            logits = clf_model(**inputs).logits
        probs   = torch.softmax(logits, dim=-1)[0]
        pred_id = probs.argmax().item()
        img_labels.append(id2label[pred_id])
        img_scores.append(float(probs[pred_id]))
    all_clf_labels.append(img_labels)
    all_clf_scores.append(img_scores)

total_crops = sum(len(l) for l in all_clf_labels)
print(f'Classified {total_crops} crops across {len(all_clf_labels)} images')

In [ ]:
## Confidence threshold sweep — mAP table

import torch, torchvision
import numpy as np
from scipy.optimize import linear_sum_assignment
from supervision.metrics import MeanAveragePrecision


def match_detections(gt_det, pred_det, iou_threshold=0.5):
    if len(gt_det) == 0 and len(pred_det) == 0:
        return np.array([], dtype=int), np.array([], dtype=int), np.array([], dtype=int)
    if len(gt_det) == 0:
        return np.array([], dtype=int), np.array([], dtype=int), np.arange(len(pred_det), dtype=int)
    if len(pred_det) == 0:
        return np.array([], dtype=int), np.array([], dtype=int), np.array([], dtype=int)
    iou = torchvision.ops.box_iou(
        torch.from_numpy(gt_det.xyxy), torch.from_numpy(pred_det.xyxy)
    ).numpy()
    gi, pi = linear_sum_assignment(-iou)
    matched_gt, matched_pred, matched_set = [], [], set()
    for g, p in zip(gi, pi):
        if iou[g, p] >= iou_threshold:
            matched_gt.append(g)
            matched_pred.append(p)
            matched_set.add(p)
    fp = [j for j in range(len(pred_det)) if j not in matched_set]
    return np.array(matched_gt, dtype=int), np.array(matched_pred, dtype=int), np.array(fp, dtype=int)


# mAP computed once on all detections at base threshold so the full PR curve is used
map_result = MeanAveragePrecision().update(all_raw_preds, all_gt).compute()
print(f'mAP@50     : {map_result.map50:.4f}')
print(f'mAP@50:95  : {map_result.map50_95:.4f}')
print()

rows = []
for thresh in SWEEP_THRESHOLDS:
    filtered = []
    for det in all_raw_preds:
        if len(det) == 0 or det.confidence is None:
            filtered.append(sv.Detections.empty())
        else:
            filtered.append(det[det.confidence >= thresh])

    tp = fp = fn = 0
    for gt_d, pred_d in zip(all_gt, filtered):
        mg, _, fpi = match_detections(gt_d, pred_d)
        tp += len(mg)
        fp += len(fpi)
        fn += len(gt_d) - len(mg)

    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

    rows.append(dict(
        Threshold = thresh,
        Precision = prec,
        Recall    = rec,
        F1        = f1,
        TP        = tp,
        FP        = fp,
        FN        = fn,
        Dets      = sum(len(d) for d in filtered),
    ))

hdr = (f"{'Thresh':>7}  {'Prec':>7}  {'Rec':>7}  {'F1':>7}  "
       f"{'TP':>6}  {'FP':>6}  {'FN':>6}  {'Dets':>6}")
sep = '-' * len(hdr)
print(sep)
print(hdr)
print(sep)
for r in rows:
    print(f"  {r['Threshold']:>5.2f}  {r['Precision']:>7.4f}  {r['Recall']:>7.4f}  {r['F1']:>7.4f}  "
          f"{r['TP']:>6d}  {r['FP']:>6d}  {r['FN']:>6d}  {r['Dets']:>6d}")
print(sep)

# ── Sanity checks ──────────────────────────────────────────────────────────
print('\nSanity checks:')
total_gt = sum(len(g) for g in all_gt)
ok = True

for r in rows:
    if r['TP'] + r['FP'] != r['Dets']:
        print(f"  FAIL thresh={r['Threshold']:.2f}: TP+FP={r['TP']+r['FP']} != Dets={r['Dets']}")
        ok = False
    if r['TP'] + r['FN'] != total_gt:
        print(f"  FAIL thresh={r['Threshold']:.2f}: TP+FN={r['TP']+r['FN']} != total_gt={total_gt}")
        ok = False

dets = [r['Dets'] for r in rows]
for i in range(1, len(dets)):
    if dets[i] > dets[i - 1]:
        print(f"  FAIL detections increased from thresh={rows[i-1]['Threshold']:.2f} "
              f"({dets[i-1]}) to thresh={rows[i]['Threshold']:.2f} ({dets[i]})")
        ok = False

if ok:
    print(f'  All checks passed  (total GT boxes = {total_gt})')